# NumPy Neural Network Demo

This notebook demonstrates the Phase 3 NumPy Neural Network training on a small subset of the MNIST dataset.

In [ ]:
import sys
import pathlib
import numpy as np
import matplotlib.pyplot as plt

# Add backend to path
sys.path.insert(0, str(pathlib.Path().resolve().parent))

from app.model.network import NeuralNetwork
from app.dataset.mnist_loader import load_mnist

## 1. Load Data
We load a tiny subset (e.g. 1000 samples) to train quickly in this demo.

In [ ]:
X_train, y_train = load_mnist(split='train', subset_size=1000)
X_test, y_test = load_mnist(split='test', subset_size=200)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

## 2. Initialize Network
The network is 784 -> 128 (ReLU) -> 64 (ReLU) -> 10 (Softmax).

In [ ]:
nn = NeuralNetwork(seed=42)

## 3. Train

In [ ]:
epochs = 10
batch_size = 32
learning_rate = 0.1
num_samples = len(X_train)

history = {'loss': [], 'accuracy': []}

for epoch in range(epochs):
    # Shuffle
    indices = np.random.permutation(num_samples)
    X_shuffled = X_train[indices]
    y_shuffled = y_train[indices]
    
    epoch_loss = 0
    batches = 0
    for i in range(0, num_samples, batch_size):
        X_batch = X_shuffled[i:i+batch_size]
        y_batch = y_shuffled[i:i+batch_size]
        
        # One-hot encoding
        y_one_hot = np.zeros((len(y_batch), 10))
        y_one_hot[np.arange(len(y_batch)), y_batch] = 1
        
        # Forward pass
        probs = nn.forward(X_batch, training=True)
        loss = nn.loss_fn.forward(probs, y_one_hot)
        epoch_loss += loss
        batches += 1
        
        # Backward pass
        d_loss = nn.loss_fn.backward()
        nn.backward(d_loss)
        
        # Update weights
        nn.update_weights(learning_rate)
        
    avg_loss = epoch_loss / batches
    
    # Evaluate on test set
    test_loss, test_acc = nn.evaluate(X_test, y_test)
    
    history['loss'].append(avg_loss)
    history['accuracy'].append(test_acc)
    
    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {avg_loss:.4f} - Test Acc: {test_acc:.4f}")

## 4. Plot Results

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history['loss'])
plt.title('Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.subplot(1, 2, 2)
plt.plot(history['accuracy'])
plt.title('Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.tight_layout()
plt.show()